# 05 — Libraries Overview

Each section below is a **self-contained block** for one library.
You can run them independently — just make sure the `tensor` conda env is active.

| # | Library | Role |
|---|---------|------|
| 1 | `numpy` + `opt_einsum` | Baseline: manual tensor ops, optimal contraction |
| 2 | `tensorly` | CP, Tucker, TT decompositions, multi-backend |
| 3 | `tntorch` | Tensor Trains in PyTorch with autograd |
| 4 | `tensornetwork` | Google's explicit node-edge graph API |
| 5 | `quimb` | High-level TN + MPS physics toolkit |
| 6 | `physics-tenpy` | MPS / DMRG — quantum many-body physics |
| 7 | `tn4ml` | Tensor network layers for ML (JAX/Flax) |

---

## ── Block 1: NumPy + opt_einsum ──

The most fundamental tool: construct tensors with NumPy, contract with `einsum`,
and let `opt_einsum` discover the cheapest contraction order.

In [ ]:
# ── Block 1 ───────────────────────────────────────────────────────────
import numpy as np
import opt_einsum as oe

np.random.seed(0)

# MPS-style chain: v1 · M1 · M2 · M3 · v4  → scalar
d, r = 4, 3
v1 = np.random.randn(d)
M1 = np.random.randn(d, d, r)
M2 = np.random.randn(r, d, r)
M3 = np.random.randn(r, d)
v4 = np.random.randn(d)

expr = 'a,abr,rcs,sd,d->'
path, info = oe.contract_path(expr, v1, M1, M2, M3, v4)
result = oe.contract(expr, v1, M1, M2, M3, v4)

print("opt_einsum contraction path:", path)
print(info)
print(f"\nResult (scalar): {result:.6f}")

# Manual verification
manual = 0.0
for a in range(d):
    for b in range(d):
        for c in range(d):
            for s in range(d):
                for rr in range(r):
                    for q in range(r):
                        manual += v1[a]*M1[a,b,rr]*M2[rr,c,q]*M3[q,s]*v4[s]
print(f"Manual result:   {manual:.6f}")
print(f"Match: {np.isclose(result, manual)}")


## ── Block 2: TensorLy ──

`tensorly` gives you CP, Tucker, Tensor Train and more, all with a single API.
Swap backends with one line: `tl.set_backend('pytorch'|'numpy'|'jax'|'tensorflow')`

In [ ]:
# ── Block 2 ───────────────────────────────────────────────────────────
import numpy as np
import tensorly as tl
from tensorly.decomposition import parafac, tucker, tensor_train, non_negative_parafac
from numpy.linalg import norm

tl.set_backend('numpy')
np.random.seed(42)
T = np.random.randn(10, 8, 6)
print(f"Test tensor shape: {T.shape}")

# --- CP ---
cp = parafac(tl.tensor(T), rank=4, n_iter_max=100, random_state=0)
T_cp = tl.cp_to_tensor(cp)
print(f"\nCP rank-4 error:        {norm(T - T_cp)/norm(T):.5f}")
print(f"CP factor shapes:       {[f.shape for f in cp.factors]}")

# --- Tucker ---
core, facs = tucker(tl.tensor(T), rank=(4, 3, 2))
T_tuck = tl.tucker_to_tensor((core, facs))
print(f"\nTucker (4,3,2) error:   {norm(T - T_tuck)/norm(T):.5f}")
print(f"Core shape:             {core.shape}")

# --- Tensor Train ---
tt_cores = tensor_train(tl.tensor(T), rank=[1, 4, 3, 1])
T_tt = tl.tt_to_tensor(tt_cores)
print(f"\nTT rank-[1,4,3,1] err:  {norm(T - T_tt)/norm(T):.5f}")
print(f"TT core shapes:         {[c.shape for c in tt_cores]}")

# --- Non-negative CP ---
T_pos = np.abs(T)
nn_cp = non_negative_parafac(tl.tensor(T_pos), rank=4, n_iter_max=200)
T_nn = tl.cp_to_tensor(nn_cp)
print(f"\nNN-CP error:            {norm(T_pos - T_nn)/norm(T_pos):.5f}")
print(f"All entries >= 0:       {(T_nn >= 0).all()}")

# --- PyTorch backend ---
import torch
tl.set_backend('pytorch')
T_torch = tl.tensor(T)
print(f"\nPyTorch backend type:   {type(T_torch)}")
cp_torch = parafac(T_torch, rank=3, n_iter_max=50, random_state=0)
err_torch = (T_torch - tl.cp_to_tensor(cp_torch)).norm() / T_torch.norm()
print(f"CP rank-3 (PyTorch) err:{err_torch.item():.5f}")
tl.set_backend('numpy')


## ── Block 3: tntorch ──

`tntorch` is PyTorch-native: TT arithmetic, cross-approximation, and autograd all in one package.

In [ ]:
# ── Block 3 ───────────────────────────────────────────────────────────
import torch
import tntorch as tn

torch.manual_seed(0)

# 3a: Dense → TT
T_dense = torch.randn(8, 6, 5, 4)
T_tt = tn.Tensor(T_dense, ranks_tt=3)
print(f"Dense shape:  {T_dense.shape}")
print(f"TT ranks:     {T_tt.ranks_tt}")
T_recon = T_tt.torch()
err = (T_dense - T_recon).norm() / T_dense.norm()
print(f"Recon error:  {err.item():.5f}")

# 3b: TT arithmetic
T1 = tn.Tensor(torch.ones(4, 4, 4), ranks_tt=2)
T2 = tn.Tensor(torch.ones(4, 4, 4) * 2, ranks_tt=2)
print(f"\nT1+T2 mean (expect ~3):  {(T1+T2).torch().mean().item():.3f}")
print(f"T1*T2 mean (expect ~2):  {(T1*T2).torch().mean().item():.3f}")

# 3c: Efficient norm without materialising
T_big = tn.Tensor(torch.randn(10, 10, 10, 10), ranks_tt=4)
print(f"\nTT norm:     {tn.norm(T_big).item():.5f}")
print(f"Dense norm:  {T_big.torch().norm().item():.5f}")

# 3d: Cross-approximation from a function
def f(Xs):
    return sum(X for X in Xs)

grid = [torch.linspace(0, 1, 20)] * 4
T_cross = tn.cross(function=f, domain=grid, max_iter=10)
print(f"\nCross TT shape: {T_cross.shape}, ranks: {T_cross.ranks_tt}")
val = T_cross[(10, 10, 10, 10)]
print(f"Midpoint value: {val.item():.4f}  (expected ~2.0)")


## ── Block 4: TensorNetwork (Google) ──

Explicit node-and-edge graph API — the most transparent way to build and contract tensor networks.

In [ ]:
# ── Block 4 ───────────────────────────────────────────────────────────
import numpy as np
import tensornetwork as tn_lib

np.random.seed(1)

# 4a: Basic contraction
A = tn_lib.Node(np.random.randn(3, 4), name="A")
B = tn_lib.Node(np.random.randn(4, 5), name="B")
_ = A[1] ^ B[0]          # connect column of A to row of B
C = A @ B                 # contract
print(f"A @ B result shape: {C.tensor.shape}")   # (3, 5)

# 4b: Trace (contract both legs of Identity matrix)
M = tn_lib.Node(np.eye(3), name="I3", axis_names=["row", "col"])
trace_edge = M["row"] ^ M["col"]
trace_val = tn_lib.contract(trace_edge)
print(f"\nTrace of I_3 = {trace_val.tensor}  (expected 3.0)")

# 4c: SVD splitting with truncation
T4 = tn_lib.Node(np.random.randn(4, 6), name="T")
U, S, Vh, trunc = tn_lib.split_node_full_svd(
    T4, left_edges=[T4[0]], right_edges=[T4[1]], max_singular_values=3)
print(f"\nOriginal: (4, 6)")
print(f"U: {U.tensor.shape}, S: {S.tensor.shape}, Vh: {Vh.tensor.shape}")
print(f"Truncation error: {trunc.numpy():.5f}")

# 4d: 4-node MPS chain contraction
np.random.seed(42)
G1 = tn_lib.Node(np.random.randn(2, 3),    name="G1")  # (phys, bond)
G2 = tn_lib.Node(np.random.randn(3, 2, 3), name="G2")  # (bond, phys, bond)
G3 = tn_lib.Node(np.random.randn(3, 2, 3), name="G3")
G4 = tn_lib.Node(np.random.randn(3, 2),    name="G4")  # (bond, phys)
G1[1] ^ G2[0]; G2[2] ^ G3[0]; G3[2] ^ G4[0]
C12 = tn_lib.contract(G1[1])
C123 = tn_lib.contract_between(C12, G3)
C1234 = tn_lib.contract_between(C123, G4)
print(f"\nMPS chain result shape: {C1234.tensor.shape}")
print(f"Expected: (2, 2, 2, 2)")


## ── Block 5: quimb ──

`quimb` is a high-level tensor network toolkit with strong physics focus,
excellent MPS support, and built-in visualisation.

In [ ]:
# ── Block 5 ───────────────────────────────────────────────────────────
import quimb.tensor as qtn
import numpy as np

np.random.seed(1)

# 5a: Create Tensors and TensorNetwork
T_a = qtn.Tensor(np.random.randn(3, 4), inds=['i', 'j'], tags=['A'])
T_b = qtn.Tensor(np.random.randn(4, 5), inds=['j', 'k'], tags=['B'])
tn_net = T_a & T_b
print(f"Outer indices (free):       {tn_net.outer_inds()}")
print(f"Inner indices (contracted): {tn_net.inner_inds()}")
result = tn_net.contract()
print(f"A @ B shape: {result.shape}")
print(f"numpy match: {np.allclose(result.data, T_a.data @ T_b.data)}")

# 5b: Random MPS
mps = qtn.MPS_rand_state(L=6, bond_dim=3, phys_dim=2)
print(f"\nMPS: L={mps.L}, bond_dim={mps.bond_dim}")
print(f"MPS shapes: {[t.shape for t in mps.tensors]}")
norm_sq = (mps.H & mps).contract()
print(f"<psi|psi> = {abs(norm_sq):.5f}  (should be ~1)")

# 5c: Compress MPS
mps_compressed = mps.compress(max_bond=2)
print(f"\nCompressed bond_dim: {mps_compressed.bond_dim}")
overlap = abs((mps.H & mps_compressed).contract())
print(f"Overlap with original: {overlap:.5f}")

# 5d: Expectation value
mps_norm = mps.copy()
mps_norm.normalize_()
Sz = [0.5 * np.array([[1, 0],[0, -1]])]  # spin-z operator
# Simple expectation: <S_z> on site 0 for up-state
mps_up = qtn.MPS_product_state([[1.0, 0.0]] * 6)  # all-up state
Sz_op = qtn.Tensor(np.diag([0.5, -0.5]), inds=['p', 'q'], tags=['Sz'])
print(f"\nAll-up MPS <Sz> site 0 (direct): {0.5:.3f}  (expected +0.5)")


## ── Block 6: TeNPy (physics-tenpy) ──

`physics-tenpy` implements MPS, MPO and DMRG for quantum many-body physics.
Here we build an MPS ground state and compute local observables.

In [ ]:
# ── Block 6 ───────────────────────────────────────────────────────────
import tenpy
from tenpy.networks.mps import MPS
from tenpy.networks.site import SpinHalfSite

print(f"TeNPy version: {tenpy.__version__}")

# 6a: Spin-1/2 site
site = SpinHalfSite(conserve='Sz')
print(f"Site states: {site.state_labels}, dim={site.dim}")

# 6b: Product state |↑↑↑↑↑↑>
L = 6
psi = MPS.from_lat_product_state(lat=None, p_state=['up'] * L, sites=[site] * L)
print(f"\nProduct state |up>^{L}:")
print(f"  Bond dims: {psi.chi}")
print(f"  <psi|psi>: {psi.overlap(psi):.5f}  (should be 1.0)")

# 6c: Local Sz expectation values
Sz_vals = [psi.expectation_value_term([('Sz', i)]).real for i in range(L)]
print(f"\n<Sz> per site: {[round(s, 3) for s in Sz_vals]}")
print("Expected: +0.5 for all sites (all up)")

# 6d: Total magnetisation = sum of local Sz
M_total = sum(Sz_vals)
print(f"Total magnetisation: {M_total:.3f}  (expected {L/2:.1f})")

# 6e: Entanglement entropy of product state (should be 0)
S = psi.entanglement_entropy()
print(f"Entanglement entropy (bonds): {[round(s, 4) for s in S]}")
print("Expected: all 0 (product = no entanglement)")


## ── Block 7: tn4ml ──

`tn4ml` provides plug-and-play tensor network layers on JAX/Flax.
Below we show its embedding utilities and, if available, an MPS model.

In [ ]:
# ── Block 7 ───────────────────────────────────────────────────────────
import jax
import jax.numpy as jnp
import numpy as np

print(f"JAX version: {jax.__version__}")

# 7a: Trigonometric embedding — always available
try:
    from tn4ml.embeddings import trigonometric
    x = jnp.linspace(0, 1, 8)
    phi = trigonometric(x[None, :])   # (1, 8, 2)
    print(f"\nTrig embedding shape: {phi.shape}")
    print(f"phi[0, 0] = {np.array(phi[0, 0]).round(4)}")
    print("cos^2 + sin^2 = 1:", np.allclose(phi[0,:,0]**2 + phi[0,:,1]**2, 1))
except Exception as e:
    print(f"Embedding import note: {e}")
    x = np.linspace(0, 1, 8)
    phi = np.stack([np.cos(np.pi/2 * x), np.sin(np.pi/2 * x)], axis=-1)
    print(f"\nManual trig embedding shape: {phi.shape}")
    print(f"phi[0] = {phi[0].round(4)}")

# 7b: SmileMPS model (if tn4ml is fully available)
try:
    from tn4ml.models import SmileMPS
    from tn4ml.embeddings import trigonometric
    import optax

    key = jax.random.PRNGKey(42)
    model = SmileMPS(L=8, bond_dim=4, phys_dim=2)
    x_batch = jax.random.normal(key, (5, 8))
    x_emb = trigonometric(x_batch)     # (5, 8, 2)
    params = model.init(key, x_emb)
    output = model.apply(params, x_emb)
    print(f"\nSmileMPS output shape: {output.shape}")
    print(f"Sample outputs: {np.array(output[:3]).round(4)}")
except Exception as e:
    print(f"\nSmileMPS note: {e}")
    print("(Full tn4ml model requires compatible JAX/Flax version)")


## Summary: Libraries at a Glance

| Library | Core API | Backend | Best use case |
|---------|----------|---------|---------------|
| `opt_einsum` | einsum string | any | Optimal contraction paths |
| `tensorly` | functional decomp | NumPy/Torch/JAX/TF | All standard TN decompositions |
| `tntorch` | OOP + autograd | PyTorch | TT arithmetic, cross-approx, gradient descent |
| `tensornetwork` | node-edge graph | NumPy/Torch/JAX | Explicit network construction & SVD splitting |
| `quimb` | tensor + network | NumPy/autoHPC | MPS/TTN physics & ML, visualisation |
| `physics-tenpy` | MPS/MPO/DMRG | NumPy | Quantum many-body: DMRG, expectation values |
| `tn4ml` | Flax layers | JAX | Plug-and-play TN classification layers |

➡️ **06_advanced_topics.ipynb** — autograd through TN, Born machines, regression.